# Web Scraping for Buying Prices

In [1]:
#pip install undetected-chromedriver

Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
import random 
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd

try:
    options = uc.ChromeOptions()
    options.add_argument("start-maximized")
    driver = uc.Chrome(options=options, use_subprocess=True)
    print("Undetected ChromeDriver başarıyla başlatıldı.")
except Exception as e:
    print(f"HATA: ChromeDriver başlatılamadı. Lütfen 'pip install undetected-chromedriver --upgrade' komutu ile kütüphaneyi güncelleyin. Hata: {e}")
    exit()

data = []

for page in range(520, 550):
    try:
        # Bir önceki oturumun izlerini silerek sitenin bize hep aynı sayfayı göstermesini engelliyoruz.
        print(f"\n{page}. sayfaya geçmeden önce çerezler temizleniyor...")
        driver.delete_all_cookies()
        time.sleep(random.uniform(1, 3)) # Temizlik sonrası kısa bir nefes al

        url = f"https://www.hepsiemlak.com/istanbul-satilik?page={page}"
        driver.get(url)
        print(f"{page}. sayfa açıldı: {url}")

        # Cloudflare ve sayfa yüklemesi için rastgele bekleme
        initial_wait = random.uniform(6, 10)
        print(f"Sayfanın yüklenmesi için {initial_wait:.2f} saniye bekleniyor...")
        time.sleep(initial_wait)
        
        wait = WebDriverWait(driver, 30)
        wait.until(EC.presence_of_element_located((By.XPATH, "//ul[contains(@class, 'list-items-container')]")))
        print("İlanlar listesi başarıyla yüklendi.")

        listing_elements = driver.find_elements(By.XPATH, "//li[contains(@class, 'listing-item')]//a[contains(@class, 'card-link')]")
        unique_links = list(dict.fromkeys([el.get_attribute('href') for el in listing_elements if el.get_attribute('href')]))

        if not unique_links:
            print(f"Sayfa {page} üzerinde ilan linki bulunamadı. Sayfa atlanıyor.")
            continue

        print(f"{len(unique_links)} adet benzersiz ilan linki bulundu. Detaylar çekiliyor...")

        # Toplanan linkleri tek tek ziyaret et
        for idx, link in enumerate(unique_links):
            driver.get(link)
            print(f"  İlan {idx+1}/{len(unique_links)} işleniyor...")

            try:
                wait.until(EC.presence_of_element_located((By.XPATH, "//ul[contains(@class, 'adv-info-list')]")))
                
                price = driver.find_element(By.XPATH, "//p[contains(@class, 'price')]").text.strip()
                location_elements = driver.find_elements(By.XPATH, "//ul[contains(@class, 'detail-info-location')]/li")
                lokasyon = " - ".join([elem.text for elem in location_elements])

                ilan_verisi = {'Fiyat': price, 'Lokasyon': lokasyon, 'Link': link}
                
                features_list = driver.find_elements(By.XPATH, "//ul[contains(@class, 'adv-info-list')]/li")
                for feature_item in features_list:
                    parts = feature_item.text.split('\n')
                    if len(parts) >= 2:
                        key = parts[0].strip()
                        value = " ".join(parts[1:]).strip()
                        ilan_verisi[key] = value
                
                data.append(ilan_verisi)
                print(f"    Veri başarıyla çekildi: {price}")

            except (NoSuchElementException, TimeoutException):
                print(f"    HATA: Bu ilanın detayları okunamadı. Sonraki ilana geçiliyor.")
            
            # Her ilan arasında rastgele bir süre beklemek.
            inter_listing_wait = random.uniform(2, 5)
            time.sleep(inter_listing_wait)

    except TimeoutException:
         print(f"Sayfa {page} yüklenirken ZAMAN AŞIMI oldu. Cloudflare engellemiş olabilir.")
         cooldown_time = random.uniform(20, 40)
         print(f"Soğuma molası veriliyor: {cooldown_time:.2f} saniye...")
         time.sleep(cooldown_time)
         continue
    except Exception as e:
        print(f"Sayfa {page} işlenirken beklenmedik bir hata oluştu: {e}")
        continue

driver.quit()

if data:
    df = pd.DataFrame(data)
    df.to_excel("buying_Prices_V3_sayfa520den_sonra.xlsx", index=False)
    print(f"\nVeri çekme işlemi tamamlandı ve {len(data)} adet ilan 'buying_Prices_V3_sayfa520den_sonra.xlsx' dosyasına kaydedildi.")
else:
    print("\nHiçbir veri çekilemedi.")

Undetected ChromeDriver başarıyla başlatıldı.

520. sayfaya geçmeden önce çerezler temizleniyor...
520. sayfa açıldı: https://www.hepsiemlak.com/istanbul-satilik?page=520
Sayfanın yüklenmesi için 8.64 saniye bekleniyor...
İlanlar listesi başarıyla yüklendi.
24 adet benzersiz ilan linki bulundu. Detaylar çekiliyor...
  İlan 1/24 işleniyor...
    Veri başarıyla çekildi: 2.750.000 TL
  İlan 2/24 işleniyor...
    Veri başarıyla çekildi: 16.000.000 TL
  İlan 3/24 işleniyor...
    Veri başarıyla çekildi: 5.500.000 TL
  İlan 4/24 işleniyor...
    Veri başarıyla çekildi: 4.500.000 TL
  İlan 5/24 işleniyor...
    Veri başarıyla çekildi: 3.750.000 TL
  İlan 6/24 işleniyor...
    Veri başarıyla çekildi: 5.300.000 TL
  İlan 7/24 işleniyor...
    Veri başarıyla çekildi: 3.950.000 TL
  İlan 8/24 işleniyor...
    Veri başarıyla çekildi: 4.500.000 TL
  İlan 9/24 işleniyor...
    Veri başarıyla çekildi: 5.300.000 TL
  İlan 10/24 işleniyor...
    Veri başarıyla çekildi: 7.250.000 TL
  İlan 11/24 işleniy

In [3]:
#df = pd.DataFrame(data)
#df.to_excel("buying_Prices_V3_sayfa36dan_sonra.xlsx", index=False)
df

,Fiyat,Lokasyon,Link,İlan no,Son Güncelleme,İlan Durumu,Konut Tipi,Konut Şekli,Oda Sayısı,Banyo Sayısı,...,Eşya Durumu,Yapı Tipi,Yapının Durumu,Kullanım Durumu,Cephe,Takas,Yetkili Ofis,Kira Getirisi,Site İçerisinde,Aidat
0,1.325.000 TL,İstanbul - Esenyurt - Talatpaşa Mah.,https://www.hepsiemlak.com/istanbul-esenyurt-t...,111753-2465,19-10-2025,Satılık,Daire,Daire,2 + 1,1,...,Eşyalı Değil,Betonarme,Sıfır,Boş,"Güney, Doğu",Evet,Evet,NaN,NaN,NaN
1,1.050.000 TL,İstanbul - Beylikdüzü - Beylikdüzü OSB Mah.,https://www.hepsiemlak.com/istanbul-beylikduzu...,104770-7203,19-10-2025,Satılık,Daire,Daire,2 + 1,1,...,Eşyalı Değil,Betonarme,Sıfır,Boş,"Kuzey, Güney",Evet,Evet,NaN,NaN,NaN
2,2.050.000 TL,İstanbul - Ümraniye - Armağanevler Mah.,https://www.hepsiemlak.com/istanbul-umraniye-a...,158673-21,28-09-2025,Satılık,Daire,NaN,2 + 1,1,...,Eşyalı Değil,Betonarme,İkinci El,Boş,NaN,Hayır,NaN,20.000 TL,NaN,NaN
3,1.620.000 TL,İstanbul - Kağıthane - Çeliktepe Mah.,https://www.hepsiemlak.com/istanbul-kagithane-...,20823-5746,18-10-2025,Satılık,Daire,Daire,1 + 1,1,...,Eşyalı Değil,Betonarme,İkinci El,Kiracılı,NaN,NaN,Evet,NaN,NaN,NaN
4,2.200.000 TL,İstanbul - Fatih - Ayvansaray Mah.,https://www.hepsiemlak.com/istanbul-fatih-ayva...,83336-495,06-10-2025,Satılık,Daire,Daire,2 + 1,1,...,Eşyalı Değil,Betonarme,İkinci El,Boş,NaN,Hayır,Evet,25.000 TL,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,13.500.000 TL,İstanbul - Ataşehir - Atatürk Mah.,https://www.hepsiemlak.com/istanbul-atasehir-a...,2137-28163,17-10-2025,Satılık,Daire,NaN,2 + 1,1,...,Eşyalı Değil,NaN,İkinci El,Boş,"Kuzey, Güney, Doğu, Batı",NaN,Evet,NaN,Kardelen 4 Sitesi,4.900 TL
737,78.000.000 TL,İstanbul - Sarıyer - İstinye Mah.,https://www.hepsiemlak.com/istanbul-sariyer-is...,119921-1033,15-10-2025,Satılık,Residence,NaN,3 + 1,4,...,Eşyalı Değil,NaN,NaN,Boş,"Kuzey, Güney, Doğu, Batı",Hayır,NaN,NaN,Nidapark İstinye,NaN
738,5.400.000 TL,İstanbul - Fatih - Seyyid Ömer Mah.,https://www.hepsiemlak.com/istanbul-fatih-seyy...,107215-1489,21-10-2025,Satılık,Daire,Ters Dubleks,3 + 1,1,...,Eşyalı Değil,Betonarme,İkinci El,Boş,Güney,Hayır,Evet,30.000 TL,NaN,100 TL
739,16.500.000 TL,İstanbul - Büyükçekmece - Güzelce Mah.,https://www.hepsiemlak.com/istanbul-buyukcekme...,83899-612,17-10-2025,Satılık,Villa,NaN,5 + 2,3,...,Eşyalı Değil,NaN,NaN,Kiracılı,"Güney, Doğu, Batı",Evet,NaN,NaN,NaN,200 TL
